# Lasso-MIDAS: Regularization Path and Variable Selection

**Module 05 — Machine Learning Extensions**

**Learning objectives:**
- Build a MIDAS design matrix with many monthly predictors
- Fit Lasso, Ridge, and Elastic Net MIDAS estimators
- Visualise regularization paths and compare sparsity patterns
- Select tuning parameters via time-series cross-validation
- Evaluate out-of-sample forecasting performance

**Estimated time**: 15 minutes  
**Data**: FRED monthly economic indicators → quarterly GDP growth

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LassoCV, RidgeCV, ElasticNetCV
from sklearn.linear_model import lasso_path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import warnings
warnings.filterwarnings('ignore')

# Try FRED data; fall back to local CSV if offline
try:
    import pandas_datareader.data as web
    FRED_AVAILABLE = True
except ImportError:
    FRED_AVAILABLE = False

print("Libraries loaded successfully.")

In [ ]:
section_divider("1. Load and Prepare Data")

In [ ]:
apply_course_theme()
apply_plot_theme()

## 1. Load and Prepare Data

We will use 12 monthly FRED series as high-frequency predictors for quarterly real GDP growth. These represent the typical set of indicators used in nowcasting: employment, industrial activity, surveys, and financial conditions.

In [ ]:
def load_fred_indicators(start='2000-01-01', end='2023-12-31'):
    """
    Load monthly FRED indicators and quarterly GDP growth.
    Returns monthly DataFrame and quarterly Series.
    """
    if FRED_AVAILABLE:
        try:
            tickers = {
                'INDPRO': 'Industrial Production',
                'PAYEMS': 'Nonfarm Payrolls',
                'UNRATE': 'Unemployment Rate',
                'UMCSENT': 'Consumer Sentiment',
                'RSXFS': 'Retail Sales ex-Food',
                'HOUST': 'Housing Starts',
                'DPCERA3M086SBEA': 'PCE Real',
                'NAPM': 'ISM Manufacturing',
                'GS10': '10Y Treasury Yield',
                'BAMLH0A0HYM2': 'HY Spread',
                'ICSA': 'Initial Claims',
                'PPIACO': 'PPI All Commodities',
            }
            monthly_data = web.DataReader(
                list(tickers.keys()), 'fred', start, end
            )
            monthly_data.columns = list(tickers.values())
            monthly_data = monthly_data.resample('MS').last()

            gdp_raw = web.DataReader('A191RL1Q225SBEA', 'fred', start, end)
            gdp_growth = gdp_raw['A191RL1Q225SBEA'].dropna()
            gdp_growth.index = gdp_growth.index.to_period('Q').to_timestamp()
            return monthly_data, gdp_growth
        except Exception as e:
            print(f"FRED unavailable ({e}), using synthetic data")

    # Synthetic fallback with realistic properties
    np.random.seed(42)
    dates_monthly = pd.date_range('2000-01-01', '2023-12-01', freq='MS')
    dates_quarterly = pd.date_range('2000-01-01', '2023-10-01', freq='QS')
    T_m = len(dates_monthly)
    T_q = len(dates_quarterly)

    indicators = [
        'Industrial Production', 'Nonfarm Payrolls', 'Unemployment Rate',
        'Consumer Sentiment', 'Retail Sales', 'Housing Starts',
        'PCE Real', 'ISM Manufacturing', '10Y Yield',
        'HY Spread', 'Initial Claims', 'PPI'
    ]

    monthly_data = pd.DataFrame(index=dates_monthly)
    for i, name in enumerate(indicators):
        trend = np.linspace(0, 0.5, T_m)
        cycle = 0.3 * np.sin(2 * np.pi * np.arange(T_m) / 60 + i)
        noise = 0.5 * np.random.randn(T_m)
        monthly_data[name] = trend + cycle + noise

    # GDP growth: correlated with first 3 indicators + noise
    gdp_values = []
    for q in range(T_q):
        m_idx = q * 3
        if m_idx + 2 < T_m:
            indpro_avg = monthly_data['Industrial Production'].iloc[m_idx:m_idx+3].mean()
            payems_avg = monthly_data['Nonfarm Payrolls'].iloc[m_idx:m_idx+3].mean()
            gdp = 0.5 * indpro_avg + 0.3 * payems_avg + 0.2 * np.random.randn()
        else:
            gdp = 2.0 + 0.5 * np.random.randn()
        gdp_values.append(gdp)

    gdp_growth = pd.Series(gdp_values, index=dates_quarterly, name='GDP_Growth')
    return monthly_data, gdp_growth


monthly_data, gdp_growth = load_fred_indicators()

print(f"Monthly indicators: {monthly_data.shape}")
print(f"Quarterly GDP observations: {len(gdp_growth)}")
print(f"Monthly indicators available:")
for col in monthly_data.columns:
    print(f"  - {col}")

In [ ]:
section_divider("2. Build MIDAS Design Matrix")

## 2. Build MIDAS Design Matrix

For each quarterly observation, we stack the last 12 monthly lags of each indicator as separate features. This creates a design matrix with K × 12 columns.

In [ ]:
def build_midas_design_matrix(monthly_df, quarterly_target, n_lags=12):
    """
    Build flat-stacked MIDAS design matrix.
    
    For each quarterly date, takes the last n_lags monthly observations
    of each indicator as separate features.
    
    Parameters
    ----------
    monthly_df : DataFrame — monthly indicators
    quarterly_target : Series — quarterly GDP growth
    n_lags : int — number of monthly lags per indicator
    
    Returns
    -------
    X : DataFrame — MIDAS design matrix
    y : Series — aligned quarterly target
    feature_names : list — column names
    """
    # Take first differences to stationarise
    monthly_diff = monthly_df.diff().fillna(0)
    
    feature_rows = []
    target_rows = []
    index_rows = []
    
    for date in quarterly_target.index:
        # Get monthly data available up to (but not including) this quarter
        # Quarter start: use lags from previous month
        available = monthly_diff[monthly_diff.index < date]
        
        if len(available) < n_lags:
            continue
        
        # Take last n_lags months
        window = available.tail(n_lags)
        
        # Flatten: for each column, add lags from lag1 (most recent) to lag_n
        row = {}
        for col in monthly_df.columns:
            col_safe = col.replace(' ', '_')
            vals = window[col].values[::-1]  # lag1 = most recent
            for lag_idx, val in enumerate(vals, start=1):
                row[f"{col_safe}_lag{lag_idx}"] = val
        
        feature_rows.append(row)
        target_rows.append(quarterly_target[date])
        index_rows.append(date)
    
    X = pd.DataFrame(feature_rows, index=index_rows)
    y = pd.Series(target_rows, index=index_rows, name='GDP_Growth')
    
    return X, y


X_full, y_full = build_midas_design_matrix(monthly_data, gdp_growth, n_lags=12)

# Drop any remaining NaN rows
valid = X_full.notna().all(axis=1) & y_full.notna()
X_full = X_full[valid]
y_full = y_full[valid]

print(f"MIDAS design matrix: {X_full.shape}")
print(f"Observations: {len(y_full)}")
print(f"Features: {X_full.shape[1]} ({len(monthly_data.columns)} indicators × 12 lags)")
print(f"\nFirst 5 feature names:")
for name in X_full.columns[:5]:
    print(f"  {name}")

In [ ]:
section_divider("3. Standardise and Split Data")

## 3. Standardise and Split Data

In [ ]:
# Standardise features (critical for regularized regression)
scaler = StandardScaler()
X_arr = X_full.values
y_arr = y_full.values

# In-sample: first 80%, out-of-sample: last 20%
T = len(y_arr)
train_end = int(T * 0.8)

X_train = X_arr[:train_end]
y_train = y_arr[:train_end]
X_test = X_arr[train_end:]
y_test = y_arr[train_end:]

# Fit scaler on training data only
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
X_all_s = scaler.transform(X_arr)

print(f"Training: {train_end} quarters ({y_full.index[0].year} – {y_full.index[train_end-1].year})")
print(f"Test: {T - train_end} quarters ({y_full.index[train_end].year} – {y_full.index[-1].year})")
print(f"Feature matrix shape: {X_train_s.shape}")

In [ ]:
section_divider("4. Lasso Regularization Path")

## 4. Lasso Regularization Path

The regularization path shows how each coefficient evolves as we vary `lambda` from 0 (OLS) to a large value (null model). We can see which features enter the model first (most important) and which are selected at the CV-optimal lambda.

In [ ]:
# Compute Lasso path
alphas_path, coefs_path, _ = lasso_path(
    X_train_s, y_train,
    alphas=np.logspace(-4, 0, 80),
    max_iter=10000
)

# Select optimal alpha via time-series CV
tscv = TimeSeriesSplit(n_splits=5)
lasso_cv = LassoCV(
    alphas=np.logspace(-4, 0, 50),
    cv=tscv,
    max_iter=10000,
    random_state=42
)
lasso_cv.fit(X_train_s, y_train)
alpha_opt = lasso_cv.alpha_

# Plot regularization path — colour by indicator
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: regularization path for all features
ax = axes[0]
n_indicators = len(monthly_data.columns)
colors = plt.cm.tab20(np.linspace(0, 1, n_indicators))

for feat_idx in range(coefs_path.shape[0]):
    indicator_idx = feat_idx // 12  # Which indicator
    ax.plot(np.log10(alphas_path), coefs_path[feat_idx],
            color=colors[indicator_idx], alpha=0.4, linewidth=0.7)

ax.axvline(np.log10(alpha_opt), color='red', linestyle='--',
           linewidth=2, label=f'CV optimal λ={alpha_opt:.4f}')
ax.set_xlabel('log₁₀(λ)')
ax.set_ylabel('Coefficient')
ax.set_title('Lasso Regularization Path\n(colours = indicators)')
ax.legend()

# Right: number of selected features vs lambda
ax2 = axes[1]
n_nonzero = (np.abs(coefs_path) > 1e-8).sum(axis=0)
ax2.plot(np.log10(alphas_path), n_nonzero, 'b-', linewidth=2)
ax2.axvline(np.log10(alpha_opt), color='red', linestyle='--',
            linewidth=2, label=f'CV optimal')
ax2.set_xlabel('log₁₀(λ)')
ax2.set_ylabel('Number of non-zero coefficients')
ax2.set_title('Sparsity vs Regularization Strength')
ax2.legend()

plt.tight_layout()
plt.savefig('../resources/lasso_path.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Optimal lambda: {alpha_opt:.6f}")
print(f"Selected features: {np.sum(lasso_cv.coef_ != 0)} / {X_train_s.shape[1]}")

In [ ]:
section_divider("5. Fit Ridge and Elastic Net")

## 5. Fit Ridge and Elastic Net

Compare Lasso against Ridge (no selection) and Elastic Net (balanced shrinkage and selection).

In [ ]:
# Ridge with time-series CV
ridge_cv = RidgeCV(
    alphas=np.logspace(-2, 4, 50),
    cv=tscv,
    scoring='neg_mean_squared_error'
)
ridge_cv.fit(X_train_s, y_train)

# Elastic Net with time-series CV
en_cv = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
    alphas=np.logspace(-4, 0, 30),
    cv=tscv,
    max_iter=10000,
    random_state=42
)
en_cv.fit(X_train_s, y_train)

print("Model Summary:")
print(f"{'Model':<15} {'λ optimal':<15} {'l1_ratio':<12} {'Selected':<10} {'In-sample RMSE':<15}")
print("-" * 67)

models = {
    'Ridge': ridge_cv,
    'Lasso': lasso_cv,
    'ElasticNet': en_cv
}

for name, model in models.items():
    y_pred_train = model.predict(X_train_s)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    selected = np.sum(np.abs(model.coef_) > 1e-8)
    
    alpha = getattr(model, 'alpha_', getattr(model, 'alpha', 'N/A'))
    l1r = getattr(model, 'l1_ratio_', 1.0 if name == 'Lasso' else 0.0)
    
    print(f"{name:<15} {alpha:<15.6f} {l1r:<12.2f} {selected:<10} {rmse_train:<15.4f}")

## 6. Compare Coefficient Profiles

For a selected indicator, compare how Ridge, Lasso, and Elastic Net estimate the lag weights. MIDAS with smooth Beta weights should produce a monotonically declining profile; regularized methods may produce different patterns.

In [ ]:
# Show lag profiles for first 3 indicators
indicator_names = [col.replace(' ', '_') for col in monthly_data.columns[:3]]
n_lags = 12

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for ax_idx, ind_name in enumerate(indicator_names):
    ax = axes[ax_idx]
    
    # Find column indices for this indicator
    col_prefix = ind_name + '_lag'
    feat_indices = [i for i, col in enumerate(X_full.columns) if col.startswith(col_prefix)]
    
    if len(feat_indices) == 0:
        continue
    
    lags = list(range(1, len(feat_indices) + 1))
    
    for name, model, color, ls in [
        ('Ridge', ridge_cv, 'blue', '-'),
        ('Lasso', lasso_cv, 'red', '--'),
        ('ElasticNet', en_cv, 'green', '-.')
    ]:
        coefs = model.coef_[feat_indices]
        ax.plot(lags, coefs, color=color, linestyle=ls,
                marker='o', markersize=4, linewidth=1.5, label=name)
    
    ax.axhline(0, color='black', linewidth=0.5, linestyle=':')
    ax.set_title(f'{ind_name.replace("_", " ")}')
    ax.set_xlabel('Monthly lag')
    ax.set_ylabel('Coefficient')
    if ax_idx == 0:
        ax.legend()

plt.suptitle('Lag Weight Profiles: Ridge vs Lasso vs Elastic Net', y=1.02)
plt.tight_layout()
plt.savefig('../resources/lag_profiles_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("Observation: Ridge produces smooth profiles; Lasso creates sparse/choppy profiles;")
print("Elastic Net is intermediate.")

## 7. Out-of-Sample Forecast Evaluation

Evaluate all three estimators on the held-out test period using RMSE and compare to a naive AR(1) benchmark.

In [ ]:
def expanding_window_regularized(X_arr, y_arr, model_class, model_kwargs,
                                  min_train=40, scaler_class=StandardScaler):
    """
    Expanding-window evaluation with re-fitting at each step.
    Re-standardises at each origin (proper real-time simulation).
    """
    T = len(y_arr)
    forecasts = np.full(T, np.nan)
    
    for t in range(min_train, T):
        X_tr = X_arr[:t]
        y_tr = y_arr[:t]
        X_te = X_arr[t:t+1]
        
        sc = scaler_class()
        X_tr_s = sc.fit_transform(X_tr)
        X_te_s = sc.transform(X_te)
        
        model = model_class(**model_kwargs)
        model.fit(X_tr_s, y_tr)
        forecasts[t] = model.predict(X_te_s)[0]
    
    return forecasts


# Naive AR(1) benchmark
ar1_forecasts = np.full(len(y_arr), np.nan)
for t in range(1, len(y_arr)):
    # AR(1): predict next value as weighted average of last 4
    if t >= 4:
        ar1_forecasts[t] = np.mean(y_arr[t-4:t])
    else:
        ar1_forecasts[t] = y_arr[t-1]

# Expanding window for each estimator (using fixed optimal alpha from full-sample CV)
print("Running expanding-window evaluation...")
min_train = 30

ridge_ew = expanding_window_regularized(
    X_arr, y_arr, Ridge,
    {'alpha': ridge_cv.alpha_},
    min_train=min_train
)
lasso_ew = expanding_window_regularized(
    X_arr, y_arr, Lasso,
    {'alpha': lasso_cv.alpha_, 'max_iter': 10000},
    min_train=min_train
)
en_ew = expanding_window_regularized(
    X_arr, y_arr, ElasticNet,
    {'alpha': en_cv.alpha_, 'l1_ratio': en_cv.l1_ratio_, 'max_iter': 10000},
    min_train=min_train
)

# Evaluation window: last 20% of sample
eval_start = int(len(y_arr) * 0.8)
eval_mask = ~np.isnan(ridge_ew[eval_start:])
y_eval = y_arr[eval_start:][eval_mask]

print("\nOut-of-Sample Forecast Evaluation:")
print(f"{'Model':<15} {'RMSE':<10} {'MAE':<10} {'Bias':<10}")
print("-" * 45)

result_models = {
    'AR(1)': ar1_forecasts[eval_start:][eval_mask],
    'Ridge': ridge_ew[eval_start:][eval_mask],
    'Lasso': lasso_ew[eval_start:][eval_mask],
    'ElasticNet': en_ew[eval_start:][eval_mask],
}

for name, preds in result_models.items():
    errors = y_eval - preds
    rmse = np.sqrt(np.mean(errors**2))
    mae = np.mean(np.abs(errors))
    bias = np.mean(errors)
    print(f"{name:<15} {rmse:<10.4f} {mae:<10.4f} {bias:<10.4f}")

## 8. Visualise Forecasts vs Actuals

In [ ]:
dates = y_full.index
eval_dates = dates[eval_start:]

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(dates, y_arr, 'k-', linewidth=2, label='Actual GDP Growth', zorder=5)
ax.axvline(dates[eval_start], color='gray', linestyle=':', linewidth=1.5,
           label='Train/Test split')

plot_colors = {'Ridge': 'blue', 'Lasso': 'red', 'ElasticNet': 'green'}
for name, color in plot_colors.items():
    preds_full = {'Ridge': ridge_ew, 'Lasso': lasso_ew, 'ElasticNet': en_ew}[name]
    ax.plot(dates, preds_full, color=color, linewidth=1.2, alpha=0.7,
            linestyle='--', label=name)

ax.set_xlabel('Date')
ax.set_ylabel('GDP Growth (%)')
ax.set_title('Regularized MIDAS: Expanding-Window Forecasts vs Actuals')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../resources/regularized_midas_forecasts.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Identify Selected Indicators

For Lasso and Elastic Net, which indicators were selected? Group coefficients by indicator to find indicator-level importance.

In [ ]:
n_lags = 12
indicator_names_clean = [col.replace(' ', '_') for col in monthly_data.columns]

# Re-fit on full training set for final model inspection
X_train_s_final = scaler.fit_transform(X_arr)
lasso_final = Lasso(alpha=lasso_cv.alpha_, max_iter=10000)
lasso_final.fit(X_train_s_final, y_arr)

en_final = ElasticNet(alpha=en_cv.alpha_, l1_ratio=en_cv.l1_ratio_, max_iter=10000)
en_final.fit(X_train_s_final, y_arr)

# Compute indicator-level importance (sum of absolute coefficients)
lasso_importance = {}
en_importance = {}

for k, ind_name in enumerate(indicator_names_clean):
    start_col = k * n_lags
    end_col = (k + 1) * n_lags
    lasso_importance[ind_name] = np.sum(np.abs(lasso_final.coef_[start_col:end_col]))
    en_importance[ind_name] = np.sum(np.abs(en_final.coef_[start_col:end_col]))

# Sort by Lasso importance
lasso_sorted = dict(sorted(lasso_importance.items(), key=lambda x: x[1], reverse=True))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, importance_dict, title in [
    (axes[0], lasso_sorted, 'Lasso'),
    (axes[1], en_importance, 'Elastic Net')
]:
    names = list(importance_dict.keys())
    values = list(importance_dict.values())
    colors_bar = ['steelblue' if v > 0 else 'lightgray' for v in values]
    
    ax.barh(range(len(names)), values, color=colors_bar)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels([n.replace('_', ' ') for n in names], fontsize=9)
    ax.set_xlabel('Sum of |coefficients|')
    ax.set_title(f'{title}: Indicator-Level Importance')

plt.tight_layout()
plt.savefig('../resources/indicator_importance.png', dpi=150, bbox_inches='tight')
plt.show()

selected_lasso = [k for k, v in lasso_importance.items() if v > 0]
selected_en = [k for k, v in en_importance.items() if v > 0]
print(f"Lasso selected {len(selected_lasso)}/{len(indicator_names_clean)} indicators: {selected_lasso}")
print(f"ElasticNet selected {len(selected_en)}/{len(indicator_names_clean)} indicators: {selected_en}")

## 10. Summary

In this notebook we:

1. Built a MIDAS design matrix with 12 monthly indicators × 12 lags each (144 features)
2. Fitted Lasso, Ridge, and Elastic Net with time-series cross-validation
3. Visualised the Lasso regularization path — showing how coefficients enter/exit as lambda varies
4. Compared lag weight profiles: Ridge produces smooth profiles, Lasso creates sparse/choppy ones
5. Evaluated out-of-sample RMSE with expanding-window forecasting
6. Identified which indicators each estimator selected

**Key takeaways:**
- Lasso performs indicator selection but may produce non-smooth lag profiles
- Ridge shrinks uniformly — handles collinearity but gives all indicators some weight
- Elastic Net balances both, often performing best out-of-sample
- Always use time-series CV (expanding window) for tuning — never random k-fold

**Next**: `02_xgboost_vs_midas.ipynb` — compare nonlinear tree-based nowcasting against MIDAS regression.

In [ ]:
key_takeaways(["- Lasso performs indicator selection but may produce non-smooth lag profiles", "Ridge shrinks uniformly — handles collinearity but gives all indicators some wei", "Elastic Net balances both, often performing best out-of-sample", "Always use time-series CV (expanding window) for tuning — never random k-fold"])